In [1]:
!pip install -q PyPDF2 requests groq
print("✅ Done")

✅ Done


In [ ]:
import os

# ── Paste your keys here ──────────────────────────────────────
GROQ_API_KEY = "your_groq_api_key_here"   # from console.groq.com
# ─────────────────────────────────────────────────────────────

os.environ["GROQ_API_KEY"] = GROQ_API_KEY
print("✅ Keys configured")

✅ Keys configured


In [3]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os

# Copy model files from Drive to Colab
os.makedirs('/content/models', exist_ok=True)
shutil.copy(
    '/content/drive/MyDrive/academic-paper-intelligence/textcnn_trained.pth',
    '/content/models/textcnn_trained.pth'
)
shutil.copy(
    '/content/drive/MyDrive/academic-paper-intelligence/vocab.json',
    '/content/models/vocab.json'
)
print("✅ Model files loaded from Google Drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Model files loaded from Google Drive


In [4]:
%%writefile /content/config.py
import os

GEMINI_MODEL = "groq/llama-3.3-70b-versatile"
LOGS_DIR     = "/content/logs"
OUTPUTS_DIR  = "/content/outputs"
MODEL_SAVE_PATH = "/content/models/textcnn_trained.pth"
MAX_SEQ_LEN  = 256
VOCAB_SIZE   = 30000
EMBED_DIM    = 128
NUM_FILTERS  = 100
FILTER_SIZES = [2, 3, 4]
DROPOUT      = 0.5
NUM_FIELDS   = 11
NUM_NOVELTY  = 4
NOVELTY_CONFIDENCE_THRESHOLD = 0.65

FIELD_LABELS = {
    0: "Computer Science",     1: "Economics",
    2: "Electrical Engineering", 3: "Mathematics",
    4: "Physics",              5: "Quantitative Biology",
    6: "Quantitative Finance", 7: "Statistics",
    8: "CS - Other",           9: "Physics - Other",
    10: "Math - Other",
}

NOVELTY_LABELS = {
    0: "Incremental", 1: "Moderate",
    2: "High",        3: "Breakthrough",
}

SEMANTIC_SCHOLAR_BASE_URL = "https://api.semanticscholar.org/graph/v1"
SEMANTIC_SCHOLAR_MAX_RESULTS = 5

Overwriting /content/config.py


In [5]:
import os, json, re, datetime, torch, requests
import PyPDF2

# ── Upload using files.upload() with error handling ──────────
from google.colab import files as colab_files
import io

print("📄 Click 'Choose Files' and select your PDF:")

try:
    uploaded = colab_files.upload()
    if not uploaded:
        raise ValueError("No file selected")
    pdf_path = list(uploaded.keys())[0]
    print(f"✅ Uploaded: {pdf_path}")

except Exception as e:
    # Fallback: manual path entry
    print(f"Upload failed ({e})")
    print("\nAlternative: Upload via the 📁 Files panel on the left sidebar")
    print("Then type your filename below:")
    pdf_path = input("Enter PDF filename (e.g. paper.pdf): ").strip()
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"❌ '{pdf_path}' not found. Please upload it first.")

# ── Extract text ──────────────────────────────────────────────
def extract_pdf(path):
    text = ""
    with open(path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        num_pages = len(reader.pages)
        for page in reader.pages:
            extracted = page.extract_text()
            if extracted:
                text += extracted + "\n"

    abstract = ""
    match = re.search(r'(?i)abstract[:\s]*(.*?)(?=\n\n|\n1\.|\nIntroduction)', text, re.DOTALL)
    if match:
        abstract = match.group(1).strip().replace('-\n','').replace('\n',' ')[:2000]
    else:
        abstract = text[:1000]

    title = "Unknown Title"
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    for line in lines[:10]:
        if 10 < len(line) < 200:
            title = line
            break

    refs = re.findall(r'\[\d+\]', text)

    return {
        "title":          title,
        "abstract":       abstract,
        "full_text":      text[:10000],
        "classify_input": abstract if abstract else text[:1500],
        "num_pages":      num_pages,
        "references":     refs,
    }

paper = extract_pdf(pdf_path)
print(f"\n📖 Title:  {paper['title']}")
print(f"   Pages:  {paper['num_pages']}")
print(f"   Refs:   {len(paper['references'])} found")
print(f"   Abstract: {paper['abstract'][:150]}...")

📄 Click 'Choose Files' and select your PDF:


Upload failed (RangeError: Maximum call stack size exceeded.)

Alternative: Upload via the 📁 Files panel on the left sidebar
Then type your filename below:
Enter PDF filename (e.g. paper.pdf): Rapport_mini_projet1.pdf

📖 Title:  Mini-Projet 1 — Vision Commerciale Consolidée  |  Cloud Data Warehouse
   Pages:  8
   Refs:   0 found
   Abstract: Mini-Projet 1 — Vision Commerciale Consolidée  |  Cloud Data Warehouse 
1 / 8     MINI-PROJET 1 Vision Commerciale Consolidée Module : Cloud Data Ware...


In [6]:
import sys, torch, json
import torch.nn as nn
import torch.nn.functional as F

# ── TextCNN defined inline (no separate file needed) ──────────
class TextCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(30000, 128, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(128, 100, kernel_size=fs)
            for fs in [2, 3, 4]
        ])
        self.dropout = nn.Dropout(0.5)
        feat = 100 * 3  # 300
        self.field_head = nn.Sequential(
            nn.Linear(feat, 128), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(128, 11)
        )
        self.novelty_head = nn.Sequential(
            nn.Linear(feat, 64), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(64, 4)
        )

    def forward(self, x):
        e = self.embedding(x).permute(0, 2, 1)
        pooled = []
        for conv in self.convs:
            c = F.relu(conv(e))
            c = F.max_pool1d(c, c.size(2)).squeeze(2)
            pooled.append(c)
        feat = self.dropout(torch.cat(pooled, dim=1))
        return self.field_head(feat), self.novelty_head(feat)

# ── Load vocabulary ───────────────────────────────────────────
with open('/content/models/vocab.json') as f:
    word2idx = json.load(f)

def encode(text, max_len=256):
    tokens = text.lower().split()[:max_len]
    ids = [word2idx.get(t, 1) for t in tokens]
    ids += [0] * (max_len - len(ids))
    return ids

# ── Labels ────────────────────────────────────────────────────
FIELD_LABELS = {
    0: "Computer Science",      1: "Economics",
    2: "Electrical Engineering", 3: "Mathematics",
    4: "Physics",               5: "Quantitative Biology",
    6: "Quantitative Finance",  7: "Statistics",
    8: "CS - Other",            9: "Physics - Other",
    10: "Math - Other",
}
NOVELTY_LABELS = {0: "Incremental", 1: "Moderate", 2: "High", 3: "Breakthrough"}

# ── Load model ────────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model = TextCNN().to(device)
model.load_state_dict(torch.load(
    '/content/models/textcnn_trained.pth',
    map_location=device
))
model.eval()
print("✅ TextCNN model loaded")

# ── Classify ──────────────────────────────────────────────────
with torch.no_grad():
    ids = torch.tensor([encode(paper['classify_input'])]).to(device)
    f_out, n_out = model(ids)
    f_probs = torch.softmax(f_out, dim=1)
    n_probs = torch.softmax(n_out, dim=1)
    f_conf, f_idx = f_probs.max(dim=1)
    n_conf, n_idx = n_probs.max(dim=1)

classification = {
    "field":                FIELD_LABELS[f_idx.item()],
    "field_confidence":     round(f_conf.item(), 3),
    "novelty":              NOVELTY_LABELS[n_idx.item()],
    "novelty_confidence":   round(n_conf.item(), 3),
    "requires_human_review": f_conf.item() < 0.35,
}

print(f"\n🧠 TextCNN Preliminary Classification:")
print(f"   Field:   {classification['field']} ({classification['field_confidence']*100:.1f}% confidence)")
print(f"   Novelty: {classification['novelty']} ({classification['novelty_confidence']*100:.1f}% confidence)")
if classification['requires_human_review']:
    print(f"   ⚠️  Low confidence — Classifier Agent will correct this")

Using device: cuda
✅ TextCNN model loaded

🧠 TextCNN Preliminary Classification:
   Field:   Economics (23.2% confidence)
   Novelty: Breakthrough (82.5% confidence)
   ⚠️  Low confidence — Classifier Agent will correct this


In [7]:
print("\n" + "="*60)
print("🛑  HUMAN REVIEW CHECKPOINT")
print("="*60)
print(f"  Paper:   {paper['title'][:55]}")
print(f"  Field:   {classification['field']} ({classification['field_confidence']*100:.1f}%)")
print(f"  Novelty: {classification['novelty']} ({classification['novelty_confidence']*100:.1f}%)")
print("="*60)
print("  The Classifier Agent will verify and correct the field.")
response = input("\n  Type 'yes' to proceed or 'no' to stop: ").strip().lower()

if response != 'yes':
    print("❌ Pipeline stopped.")
else:
    print("✅ Approved — running full analysis...")


🛑  HUMAN REVIEW CHECKPOINT
  Paper:   Mini-Projet 1 — Vision Commerciale Consolidée  |  Cloud
  Field:   Economics (23.2%)
  Novelty: Breakthrough (82.5%)
  The Classifier Agent will verify and correct the field.

  Type 'yes' to proceed or 'no' to stop: yes
✅ Approved — running full analysis...


In [8]:
if response == 'yes':
    from groq import Groq
    import time

    client = Groq(api_key=os.environ["GROQ_API_KEY"])

    def call_agent(role, task, context=""):
        messages = []
        if context:
            messages.append({"role": "user", "content": f"Context from previous analysis:\n{context}"})
            messages.append({"role": "assistant", "content": "Understood, I will use this context."})
        messages.append({"role": "user", "content": task})

        for attempt in range(3):
            try:
                response_obj = client.chat.completions.create(
                    model="llama-3.3-70b-versatile",
                    messages=[{"role": "system", "content": role}] + messages,
                    max_tokens=1000,
                    temperature=0.3,
                )
                return response_obj.choices[0].message.content
            except Exception as e:
                if '429' in str(e) or 'rate' in str(e).lower():
                    print(f"   Rate limit — waiting 30s...")
                    time.sleep(30)
                else:
                    raise
        return "Error generating response"

    # Search related papers
    try:
        resp = requests.get(
            "https://api.semanticscholar.org/graph/v1/paper/search",
            params={"query": paper['title'], "limit": 5,
                    "fields": "title,authors,year,citationCount"},
            timeout=10
        )
        related = resp.json().get("data", [])
        related_str = "\n".join([
            f"- {p['title']} ({p.get('year','?')}) — {p.get('citationCount',0)} citations"
            for p in related[:5]
        ]) if related else "No related papers found."
    except:
        related_str = "Semantic Scholar unavailable."

    print("🤖 Agent 1 — Classifier Agent...")
    task1_result = call_agent(
        role="You are an expert research paper classifier. Always start your response with 'Corrected Field: [field name]'.",
        task=f"""Paper: {paper['title']}
Abstract: {paper['abstract']}
TextCNN preliminary field: {classification['field']} ({classification['field_confidence']*100:.1f}% confidence — may be wrong)
Determine the TRUE research field from the abstract. Start with 'Corrected Field: [name]'.
Assess novelty level. List 3 key technical terms."""
    )
    print("   ✅ Done")

    print("🤖 Agent 2 — Extraction Agent...")
    task2_result = call_agent(
        role="You are an expert research paper analyst. Extract structured information precisely.",
        task=f"""Paper: {paper['title']}
Use the corrected field from this classification: {task1_result[:200]}
Abstract: {paper['abstract']}
Excerpt: {paper['full_text'][:2000]}
Extract:
1. Research Question (2-3 sentences)
2. Methodology
3. Key Results (3 findings)
4. Datasets/Benchmarks used
5. Limitations (2-3)
6. Reproducibility""",
        context=task1_result
    )
    print("   ✅ Done")

    print("🤖 Agent 3 — Citation Agent...")
    task3_result = call_agent(
        role="You are an expert citation and literature analyst.",
        task=f"""Paper: {paper['title']}
References found: {len(paper['references'])}
Related papers via Semantic Scholar:
{related_str}
Analyze:
1. Citation Coverage
2. Missing Citations (name 2-3 important ones)
3. Recency of references
4. Overall citation quality score (1-10)""",
        context=task1_result + "\n\n" + task2_result
    )
    print("   ✅ Done")

    print("🤖 Agent 4 — Critique Agent...")
    task4_result = call_agent(
        role="You are a senior peer reviewer for top academic journals. Write precise, evidence-backed reviews.",
        task=f"""Write a complete peer review for: {paper['title']}
Use the corrected field (NOT '{classification['field']}').

Write exactly these sections:
## Summary
## Strengths
## Weaknesses
## Citation Assessment
## Reproducibility Score
## Novelty Assessment (agree/disagree with '{classification['novelty']}')
## Final Verdict
**[Accept / Major Revision / Reject]**
(2-3 sentences justifying verdict)""",
        context=task1_result + "\n\n" + task2_result + "\n\n" + task3_result
    )
    print("   ✅ Done")

    result_str = f"{task1_result}\n\n{task2_result}\n\n{task3_result}\n\n{task4_result}"

    print("\n" + "="*60)
    print("✅ PEER REVIEW COMPLETE")
    print("="*60)
    print(task4_result)

🤖 Agent 1 — Classifier Agent...
   ✅ Done
🤖 Agent 2 — Extraction Agent...
   ✅ Done
🤖 Agent 3 — Citation Agent...
   ✅ Done
🤖 Agent 4 — Critique Agent...
   ✅ Done

✅ PEER REVIEW COMPLETE
## Summary
This project, "Mini-Projet 1 — Vision Commerciale Consolidée | Cloud Data Warehouse", aims to provide a consolidated commercial vision for sales management by designing a cloud data warehouse using Oracle databases, specifically the SH (Sales History) and OE (Order Entry) schemas, with HR for geographical dimension. The project utilizes SQL analytics to create views on a cloud server and analyzes data to provide a comprehensive view of sales performance. The outcome includes four SQL analytical views, five key performance indicators (KPIs), and six functional test cases to validate the results.

## Strengths
The project demonstrates a clear understanding of cloud data warehouse concepts and the application of SQL analytics for data analysis. The use of Oracle databases and specific schemas 

In [11]:
from google.colab import files as colab_files

if response == 'yes':
    import re, datetime

    ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    md_path = f"/content/review_{ts}.md"

    # Extract corrected field
    field_match = re.search(r'Corrected Field[:\s]+([^\n\.]{5,60})', result_str)
    corrected_field = field_match.group(1).strip() if field_match else classification['field']

    for known in ["Internet of Things", "Mathematical Physics", "Computer Science",
                   "Machine Learning", "Theoretical Physics", "Pure Mathematics",
                   "Electrical Engineering", "Data Science", "Cloud Computing"]:
        if known.lower() in result_str.lower():
            corrected_field = known
            break

    with open(md_path, 'w') as f:
        f.write(f"# Peer Review: {paper['title']}\n\n")
        f.write(f"**Generated:** {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}\n\n")
        f.write(f"**Field:** {corrected_field}\n\n")
        f.write(f"**Novelty:** {classification['novelty']} ({classification['novelty_confidence']*100:.1f}%)\n\n")
        f.write("---\n\n")
        f.write(result_str)

    print(f"\n📄 Report saved: {md_path}")
    colab_files.download(md_path)
    print("✅ Report downloaded to your computer")


📄 Report saved: /content/review_20260524_121433.md


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Report downloaded to your computer
